In [3]:
"""
verificacion_parquet.ipynb
Validación técnica y control de calidad del dataset particionado. 
Verifica integridad de tipos, nulos y duplicados tras la migración a parquet particionado.
"""

import pandas as pd

# Lectura del dataset particionado (todas las particiones fecha=YYYY-MM-DD/)
df = pd.read_parquet("../data/processed")

# fecha viene como Categorical desde particiones — convertir a string
df["fecha"] = df["fecha"].astype(str)

print(f"El shape del dataset particionado es {df.shape} \n")
print(df.dtypes)
print("\n")
print(f"La cantidad de fechas (particiones) es {df['fecha'].nunique()} \n")
print(f"Rango de fechas: {df['fecha'].min()} → {df['fecha'].max()} \n")
print(f"La cantidad de columnas con nulos es:\n{df.isnull().sum()} \n")
print(df["marca_propia"].value_counts())
print("\n")
print(f"La cantidad de duplicados es {df.duplicated(subset=['referencia', 'fecha']).sum()}")

El shape del dataset particionado es (668625, 19) 

url                              object
referencia                        int64
categoria                        object
subcategoria                     object
titulo                           object
formato                          object
precio_por_medida               float64
unidad_medida                    object
precio_anterior                 float64
precio_actual                   float64
unidad_precio                    object
divisa                           object
imagen_principal                 object
imagenes_secundarias             object
timestamp                datetime64[ns]
es_marca_propia                    bool
marca_propia                     object
tiene_precio_anterior              bool
fecha                            object
dtype: object


La cantidad de fechas (particiones) es 154 

Rango de fechas: 2025-11-03 → 2026-04-28 

La cantidad de columnas con nulos es:
url                           0
referencia    

In [4]:
# Verificación de la tabla auxiliar ultimo_precio.parquet
up = pd.read_parquet("../data/state/ultimo_precio.parquet")
print(f"ultimo_precio.parquet: {up.shape}")
print(f"Productos únicos rastreados: {up['referencia'].nunique():,}\n")
print(up.head(10))

ultimo_precio.parquet: (5014, 3)
Productos únicos rastreados: 5,014

   referencia  precio_previo fecha_previo
0        1393           2.62   2026-04-28
1        1564           3.78   2026-04-28
2        1892           1.89   2026-04-28
3        1908           3.75   2026-04-28
4        2120          45.90   2026-02-01
5        2137           2.20   2026-04-28
6        2169           2.20   2026-04-28
7        2179           4.10   2026-04-28
8        2185           4.60   2026-04-28
9        2228           6.24   2026-04-28


In [5]:
# Test de lectura filtrada (solo una partición — demuestra la ventaja del particionado)
import time

fecha_test = df["fecha"].max()

start = time.time()
df_filtrado = pd.read_parquet("../data/processed", filters=[("fecha", "=", fecha_test)])
t_filtrado = time.time() - start

start = time.time()
df_completo = pd.read_parquet("../data/processed")
t_completo = time.time() - start

print(f"Lectura filtrada ({fecha_test}): {df_filtrado.shape} en {t_filtrado:.3f}s")
print(f"Lectura completa:               {df_completo.shape} en {t_completo:.3f}s")
print(f"Speedup:                         {t_completo/t_filtrado:.1f}x más rápido")

Lectura filtrada (2026-04-28): (4311, 19) en 0.076s
Lectura completa:               (668625, 19) en 2.019s
Speedup:                         26.7x más rápido
